1. Validando a normalização da coluna contratos EY

In [1]:
import pandas as pd

from src.canonical.nact_transformer import NACTTransformer

source_path = "/app/data/raw_local/RAW/2026-06-06_0800/NACT/202301_ADMIN.xlsb"

raw_df = pd.read_excel(
    source_path,
    sheet_name="RACT",
    engine="pyxlsb",
    header=None,
    dtype=str,
)

canonical_df = NACTTransformer.transform_ey_ract(raw_df)

print(canonical_df.shape)
canonical_df.head()

(4518, 5)


,contract_number_raw,contract_number,supplier_cnpj,supplier_name,business_unit
0,ALPA5900065869,5900065869,00.973.749/0016-00,TOP SERVICE SERVICOS E SISTEMAS LTDA.,PA/MARABÁ
1,BAOVALE5900052656,5900052656,02.035.105/0001-01,UNIMAR TRANSPORTES LTDA,ES/VITÓRIA
2,CPBS5900030295,5900030295,22.320.881/0001-60,Tradimaq Ltda.,RJ/RIO DE JANEIRO
3,CPBS5900055532,5900055532,52.548.435/0001-79,JSL S/A,RJ/ITAGUAÍ (CPBS)
4,CPBS5900059303,5900059303,07.147.444/0001-01,REFRAMAX ENGENHARIA S/A,RJ/ITAGUAÍ (CPBS)


2. Testando

- ler NACT EY
- transformar para canonical_nact_contracts
- converter para Spark
- salvar em s3a://contracts/canonical/entity=nact_contracts/snapshot_date=...

In [1]:
from src.canonical.pipeline import CanonicalPipeline

pipeline = CanonicalPipeline()

output_path = pipeline.run_nact_ey_contracts(
    source_path="/app/data/raw_local/RAW/2026-06-06_0800/NACT/202301_ADMIN.xlsb",
    snapshot_date="2026-06-06_0800",
)

print(output_path)

df = pipeline.spark.read.parquet(output_path)

df.printSchema()
df.show(5)

26/06/25 18:39:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/25 18:40:17 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


s3a://contracts/canonical/entity=nact_contracts/snapshot_date=2026-06-06_0800/
root
 |-- nact_internal_id: string (nullable = true)
 |-- contract_number_raw: string (nullable = true)
 |-- contract_number: string (nullable = true)
 |-- supplier_cnpj: string (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- operation_location: string (nullable = true)
 |-- source_vendor: string (nullable = true)
 |-- source_layout: string (nullable = true)
 |-- canonical_entity: string (nullable = true)
 |-- snapshot_date: string (nullable = true)
 |-- processed_at: string (nullable = true)

+----------------+-------------------+---------------+------------------+--------------------+--------------------+-------------+-------------+----------------+---------------+--------------------+
|nact_internal_id|contract_number_raw|contract_number|     supplier_cnpj|       supplier_name|  operation_location|source_vendor|source_layout|canonical_entity|  snapshot_date|        processed_at|
+-----